# Capítulo 4 — Visão Computacional para Defesa

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Distinguir** as tarefas de **classificação**, **detecção** e **segmentação** de imagens, e formular o problema de detecção de objetos como a produção de um conjunto de **caixas delimitadoras** com classe e confiança;
2. **Compreender** as duas famílias de detectores — de **duas etapas** (R-CNN até Faster R-CNN) e de **uma etapa** (YOLO, SSD) — e a troca fundamental entre **acurácia e latência** que as separa;
3. **Dominar e implementar** os conceitos próprios da detecção — **IoU**, **caixas âncora**, **supressão não-máxima** e a avaliação por **mAP**;
4. **Explicar** o rastreamento multiobjetivo pelo paradigma de **rastreamento por detecção**, com predição de movimento (**filtro de Kalman**) e associação de dados (**algoritmo húngaro**);
5. **Reconhecer** as particularidades dos sensores de defesa — imagens **SAR**, de **satélite** e de **VANT** — e os desafios de *deslocamento de domínio* e escassez de rótulos na aplicação operacional.

> O Módulo I construiu o motor: dos modelos clássicos ao aprendizado profundo e, em particular, às redes convolucionais. O **Módulo II** põe esse motor a serviço da **percepção** — a extração de sentido operacional de sinais brutos.
>
> A CNN do Capítulo 3 respondia a uma pergunta simples — *o que há nesta imagem?*. Em defesa, a pergunta operacional é mais exigente: ***o que* há, *onde*, *quantos*, e *para onde se movem*?** Localizar e identificar múltiplos objetos, segui-los ao longo do tempo e fazê-lo sobre sensores próprios do domínio é o conteúdo deste capítulo. O *Project Maven* é o seu emblema — e mantemos, também aqui, a disciplina dos capítulos anteriores: comparar contra uma *baseline*, avaliar pela métrica certa e preservar o julgamento humano na decisão.

## Preparação do ambiente

O núcleo do capítulo — IoU, NMS, mAP e rastreamento — é implementado **do zero**, apenas com `numpy`, `matplotlib` e `scipy`, já presentes no Colab. As Seções 4.3 e 4.6 usam adicionalmente a biblioteca `ultralytics` (detector YOLO pré-treinado), instalada em uma célula própria mais adiante — a única parte do capítulo que requer *download* de pesos.

Como sempre, fixamos a **semente aleatória**: *dados os mesmos dados, os mesmos resultados*.

In [ ]:
# Preparacao do ambiente: importacoes e semente de reprodutibilidade
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy

SEMENTE = 0
np.random.seed(SEMENTE)

print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"scipy      {scipy.__version__}")
print("Ambiente pronto.")

---
## 4.1 Da classificação à detecção

A visão computacional organiza-se em três tarefas de dificuldade crescente. A **classificação** atribui um rótulo à imagem inteira (*há um navio*). A **detecção** vai além: localiza cada objeto por uma **caixa delimitadora** (*bounding box*) e o classifica (*há um navio aqui, e um segundo ali*). A **segmentação** refina ainda mais, rotulando cada *pixel*. Para a maioria das aplicações operacionais — contagem, geolocalização, rastreamento — **a detecção é o nível adequado**.

### 4.1.1 O problema da detecção

Formalmente, um detector recebe uma imagem e devolve um **conjunto de predições**, cada uma composta por uma *caixa*, uma *classe* e um *escore de confiança*. A caixa é usualmente descrita por quatro números — os cantos $(x_1, y_1, x_2, y_2)$ ou o centro e as dimensões $(x, y, w, h)$. Duas dificuldades distinguem a detecção da classificação: o **número de objetos é variável e desconhecido**, e a **qualidade da localização** passa a importar tanto quanto a do rótulo.

A célula abaixo torna a distinção visual, sobre uma cena sintética de vigilância marítima:

In [ ]:
# Classificacao, deteccao e segmentacao: a mesma cena, tres respostas
from matplotlib.patches import Rectangle

rng = np.random.default_rng(SEMENTE)

# Cena sintetica: "mar" ruidoso com duas "embarcacoes" retangulares
cena = rng.normal(0.35, 0.05, (100, 100))
navios = [(20, 30, 14, 6), (60, 65, 18, 7)]   # (x, y, largura, altura)
mascara = np.zeros_like(cena, dtype=bool)
for x, y, w, h in navios:
    cena[y:y + h, x:x + w] += 0.45
    mascara[y:y + h, x:x + w] = True
cena = np.clip(cena, 0, 1)

fig, eixos = plt.subplots(1, 3, figsize=(12.5, 4.2))
for eixo in eixos:
    eixo.imshow(cena, cmap="gray")
    eixo.axis("off")

eixos[0].set_title("Classificacao: 'ha navios' (imagem inteira)")
eixos[0].text(50, 95, "navios: SIM", color="yellow", fontsize=13,
              ha="center", weight="bold")

eixos[1].set_title("Deteccao: o que + ONDE (caixas)")
for x, y, w, h in navios:
    eixos[1].add_patch(Rectangle((x, y), w, h, fill=False,
                                 edgecolor="yellow", lw=2))
    eixos[1].text(x, y - 2, "navio", color="yellow", fontsize=10)

eixos[2].set_title("Segmentacao: rotulo por pixel")
sobre = np.zeros((*cena.shape, 4))
sobre[mascara] = [1.0, 0.85, 0.0, 0.55]
eixos[2].imshow(sobre)

plt.tight_layout(); plt.show()
print(f"A deteccao devolve um CONJUNTO de predicoes: {len(navios)} caixas,")
print("cada uma com classe e confianca -- numero variavel por imagem.")

**Observe:** as três tarefas respondem a perguntas diferentes. Para *contar, geolocalizar e rastrear*, é a **detecção** que entrega a informação na granularidade certa.

### 4.1.2 Interseção sobre união (IoU)

A medida que quantifica a qualidade de uma localização é a **interseção sobre união** (*Intersection over Union*, IoU). Para duas caixas $A$ (predita) e $B$ (verdadeira),

$$\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{área da sobreposição}}{\text{área da união}}.$$

A IoU vale **0** quando as caixas não se tocam e **1** quando coincidem. Convenciona-se que uma detecção está **correta** quando sua IoU com a caixa verdadeira supera um limiar (tipicamente **0,5**). É sobre esse critério que se constroem tanto a **avaliação** quanto a **supressão de duplicatas**, adiante.

> **🛡️ Contexto de defesa** — Em defesa, o *onde* pesa tanto quanto o *o quê*. Geolocalizar um alvo, contar embarcações em uma faixa marítima, medir o deslocamento de uma coluna entre dois quadros — tudo depende de **localização precisa**, não apenas de rotulagem. Uma caixa mal posicionada sobre uma imagem de satélite pode significar **centenas de metros de erro no terreno**. A IoU é, portanto, mais do que uma métrica de laboratório: é uma medida de **utilidade operacional**.

A **Listagem 4.1** implementa a IoU e a supressão não-máxima (que usaremos na Seção 4.3) — poucas linhas que são o **núcleo de qualquer detector**:

In [ ]:
# Listagem 4.1 - Intersecao sobre uniao (IoU) e supressao nao-maxima (NMS).
def iou(caixa_a, caixa_b):
    # Caixas no formato (x1, y1, x2, y2). Retorna a IoU em [0, 1].
    xa = max(caixa_a[0], caixa_b[0])
    ya = max(caixa_a[1], caixa_b[1])
    xb = min(caixa_a[2], caixa_b[2])
    yb = min(caixa_a[3], caixa_b[3])
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = (caixa_a[2] - caixa_a[0]) * (caixa_a[3] - caixa_a[1])
    area_b = (caixa_b[2] - caixa_b[0]) * (caixa_b[3] - caixa_b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0

def nms(caixas, escores, limiar=0.5):
    # Supressao nao-maxima: descarta deteccoes sobrepostas redundantes.
    ordem = list(np.argsort(escores)[::-1])   # maior confianca primeiro
    mantidas = []
    while ordem:
        i = ordem.pop(0)
        mantidas.append(i)
        # retem apenas caixas com baixa sobreposicao com a caixa i
        ordem = [j for j in ordem if iou(caixas[i], caixas[j]) < limiar]
    return mantidas

print("Funcoes iou() e nms() definidas -- o nucleo de qualquer detector.")

In [ ]:
# A IoU em quatro cenarios: da coincidencia perfeita a disjuncao
casos = [
    ("coincidentes",        (20, 20, 60, 60), (20, 20, 60, 60)),
    ("boa localizacao",     (20, 20, 60, 60), (25, 24, 65, 64)),
    ("localizacao pobre",   (20, 20, 60, 60), (45, 40, 85, 80)),
    ("disjuntas",           (20, 20, 60, 60), (65, 65, 95, 95)),
]

fig, eixos = plt.subplots(1, 4, figsize=(13, 3.6))
for eixo, (nome, A, B) in zip(eixos, casos):
    v = iou(A, B)
    for caixa, cor, rotulo in [(A, "tab:blue", "verdadeira"),
                               (B, "tab:red", "predita")]:
        eixo.add_patch(Rectangle((caixa[0], caixa[1]),
                                 caixa[2] - caixa[0], caixa[3] - caixa[1],
                                 fill=False, edgecolor=cor, lw=2.2,
                                 label=rotulo))
    situacao = "CORRETA" if v >= 0.5 else "incorreta"
    eixo.set_title(f"{nome}\nIoU = {v:.2f} ({situacao} @0.5)", fontsize=11)
    eixo.set_xlim(0, 100); eixo.set_ylim(100, 0)
    eixo.set_xticks([]); eixo.set_yticks([])
eixos[0].legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()

**Observe:** com o limiar convencional de 0,5, a "localização pobre" **não conta** como acerto — a detecção exige mais do que apontar a vizinhança certa.

---
## 4.2 Detectores de duas etapas: da R-CNN à Faster R-CNN

A primeira família de detectores profundos separa o problema em **duas etapas**: primeiro *propor* regiões que possivelmente contêm objetos; depois *classificar* cada região.

A **R-CNN** (2014) inaugurou a abordagem: um algoritmo clássico (*selective search*) propunha milhares de regiões candidatas, e uma CNN extraía feições de cada uma para classificação. O resultado era preciso, porém **lento** — a CNN reprocessava cada região independentemente. A **Fast R-CNN** acelerou-a ao computar o mapa de feições **uma única vez** para a imagem inteira e recortar dele as regiões (*RoI pooling*). A **Faster R-CNN** (2015) deu o passo final: substituiu o proponente clássico por uma **rede de proposição de regiões** (*Region Proposal Network*, RPN), treinada em conjunto com o detector. Todo o sistema passou a ser **aprendido de ponta a ponta**.

### 4.2.1 Caixas âncora

A peça que torna a RPN eficaz é a **caixa âncora** (*anchor box*). Em vez de prever caixas do nada, a rede parte de um conjunto de **caixas de referência** — de tamanhos e proporções variados, distribuídas por toda a imagem — e aprende apenas os **ajustes** (deslocamento e escala) que as alinham aos objetos, além da probabilidade de cada âncora conter de fato um objeto. Esse artifício transforma a detecção em um problema de **regressão bem-comportado** e é herdado, em variações, pela maioria dos detectores modernos.

Os detectores de duas etapas são, em geral, **os mais precisos, ao custo de maior latência**. São a escolha natural quando o tempo não é crítico e a acurácia é soberana — a análise cuidadosa de imagens de satélite de alta resolução, por exemplo.

In [ ]:
# Caixas ancora: o vocabulario de referencia de que a rede parte
escalas = [12, 24, 40]              # tamanhos (pixels)
proporcoes = [0.5, 1.0, 2.0]        # largura/altura

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.6))

# painel 1: as 9 ancoras (3 escalas x 3 proporcoes) em um unico ponto
cores = plt.cm.viridis(np.linspace(0.15, 0.9, 9))
k = 0
for s in escalas:
    for p in proporcoes:
        w, h = s * np.sqrt(p), s / np.sqrt(p)
        a1.add_patch(Rectangle((50 - w / 2, 50 - h / 2), w, h, fill=False,
                               edgecolor=cores[k], lw=1.8))
        k += 1
a1.plot(50, 50, "k+", ms=12)
a1.set_xlim(0, 100); a1.set_ylim(100, 0)
a1.set_title("9 ancoras (3 escalas x 3 proporcoes)\nem UM ponto da grade")
a1.set_xticks([]); a1.set_yticks([])

# painel 2: a grade de pontos cobre toda a imagem
for gx in range(10, 100, 16):
    for gy in range(10, 100, 16):
        a2.plot(gx, gy, "k+", ms=7, alpha=0.5)
        a2.add_patch(Rectangle((gx - 8, gy - 8), 16, 16, fill=False,
                               edgecolor="tab:blue", lw=0.7, alpha=0.5))
alvo = (58, 34, 26, 12)
a2.add_patch(Rectangle(alvo[:2], alvo[2], alvo[3], fill=False,
                       edgecolor="tab:red", lw=2.5))
a2.text(alvo[0], alvo[1] - 3, "objeto: a RPN ajusta a ancora mais proxima",
        color="tab:red", fontsize=10)
a2.set_xlim(0, 100); a2.set_ylim(100, 0)
a2.set_title("A grade de ancoras cobre toda a imagem")
a2.set_xticks([]); a2.set_yticks([])
plt.tight_layout(); plt.show()

print(f"Em cada ponto da grade, {len(escalas) * len(proporcoes)} ancoras;")
print("a rede aprende apenas DESLOCAMENTO e ESCALA -- regressao bem-comportada.")

**Observe:** a rede nunca "inventa" uma caixa do zero — ela **corrige** a âncora mais próxima. Prever ajustes pequenos é um problema muito mais fácil do que prever coordenadas absolutas.

---
## 4.3 Detectores de uma etapa: YOLO e a detecção em tempo real

Quando a **latência** importa — e em vídeo de sensor ela quase sempre importa —, a segunda família se impõe. O **YOLO** (*You Only Look Once*, 2016) abandona a etapa de proposição: uma **única passagem** da rede sobre a imagem prevê, simultaneamente, as caixas, as classes e a probabilidade de haver objeto (*objectness*) em uma grade de células. Sem o gargalo das duas etapas, o YOLO e seus sucessores alcançam **tempo real**, dezenas de quadros por segundo — a troca é uma acurácia historicamente um pouco inferior à dos detectores de duas etapas, diferença que as versões recentes reduziram muito.

### 4.3.1 Supressão não-máxima

Tanto YOLO quanto Faster R-CNN produzem, para um mesmo objeto, **várias caixas sobrepostas**. A **supressão não-máxima** (*Non-Maximum Suppression*, NMS) resolve isso com um procedimento simples: ordenam-se as caixas por confiança; mantém-se a de maior escore; descartam-se todas as que se sobrepõem a ela acima de um limiar de IoU; repete-se com as restantes. A função `nms` da Listagem 4.1 implementa exatamente isso — a célula abaixo a vê em ação:

In [ ]:
# NMS em acao: de um enxame de deteccoes a uma caixa por objeto
rng = np.random.default_rng(SEMENTE)

# Dois objetos verdadeiros; o detector emite varias caixas por objeto
objetos = [(25, 30, 55, 55), (60, 60, 90, 85)]
caixas, escores = [], []
for ox1, oy1, ox2, oy2 in objetos:
    for _ in range(6):
        dx, dy = rng.normal(0, 2.5, 2)
        de = rng.normal(0, 2.0)
        caixas.append((ox1 + dx, oy1 + dy, ox2 + dx + de, oy2 + dy + de))
        escores.append(float(rng.uniform(0.55, 0.98)))
caixas = np.array(caixas); escores = np.array(escores)

mantidas = nms(caixas, escores, limiar=0.4)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.6))
for eixo, indices, titulo in [
        (a1, range(len(caixas)), f"antes da NMS: {len(caixas)} caixas"),
        (a2, mantidas, f"depois da NMS: {len(mantidas)} caixas")]:
    for i in indices:
        x1, y1, x2, y2 = caixas[i]
        eixo.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                 edgecolor="tab:red", lw=1.6, alpha=0.8))
        eixo.text(x1 + 1, y1 + 3.5, f"{escores[i]:.2f}", fontsize=8,
                  color="tab:red")
    eixo.set_xlim(0, 110); eixo.set_ylim(110, 0)
    eixo.set_title(titulo)
    eixo.set_xticks([]); eixo.set_yticks([])
plt.tight_layout(); plt.show()

for limiar in [0.3, 0.5, 0.7]:
    n = len(nms(caixas, escores, limiar=limiar))
    print(f"limiar de IoU da NMS = {limiar} -> {n} caixas mantidas")

**Observe:** o limiar da NMS é uma calibração com dois modos de falha — **alto demais** deixa duplicatas; **baixo demais** pode fundir alvos próximos em uma única caixa (o Exercício 4.3 explora ambos).

### 4.3.2 Avaliação: precisão média (mAP)

Como avaliar um detector? A acurácia de classificação **não serve** — ela ignora a localização e o número variável de objetos. O padrão é a **precisão média** (*mean Average Precision*, mAP). Para cada classe, uma detecção conta como **verdadeiro positivo** se sua IoU com uma caixa verdadeira supera o limiar; calcula-se então a curva **precisão–revocação** (os mesmos conceitos do Capítulo 2) e a **área sob ela** — a *Average Precision* (AP). A mAP é a média das AP sobre todas as classes, frequentemente reportada em vários limiares de IoU. **É a métrica que qualquer avaliação séria de detecção deve reportar.**

> **🛡️ Contexto de defesa** — A detecção em tempo real sobre vídeo em movimento captado por VANT é precisamente o cenário central do *Project Maven*: sinalizar objetos de interesse no fluxo de imagens mais rápido do que qualquer equipe humana conseguiria varrer. Nesse regime, o **orçamento de latência** é uma restrição de projeto tão dura quanto a acurácia — e empurra a escolha para detectores de **uma etapa**, executados, com frequência, em *hardware* embarcado com energia e memória limitadas (tema do Módulo IV).

A célula abaixo implementa a AP **do zero** e a calcula sobre um detector sintético, em dois limiares de IoU:

In [ ]:
# Average Precision (AP) do zero: casamento por IoU + curva precisao-revocacao
def average_precision(gt_por_imagem, det_por_imagem, limiar_iou=0.5):
    # Casa deteccoes (por ordem de escore) com caixas verdadeiras e
    # integra a curva precisao-revocacao (padrao "all points").
    registros = []                       # (escore, e_verdadeiro_positivo)
    total_gt = sum(len(g) for g in gt_por_imagem)
    for gts, dets in zip(gt_por_imagem, det_por_imagem):
        usadas = set()
        for escore, caixa in sorted(dets, key=lambda d: -d[0]):
            melhor, melhor_iou = None, limiar_iou
            for k, g in enumerate(gts):
                if k in usadas:
                    continue
                v = iou(caixa, g)
                if v >= melhor_iou:
                    melhor, melhor_iou = k, v
            if melhor is not None:
                usadas.add(melhor)
                registros.append((escore, 1))
            else:
                registros.append((escore, 0))   # falso positivo
    registros.sort(key=lambda r: -r[0])
    tp = np.cumsum([r[1] for r in registros])
    fp = np.cumsum([1 - r[1] for r in registros])
    revocacao = tp / max(total_gt, 1)
    precisao = tp / np.maximum(tp + fp, 1e-9)
    envelope = np.maximum.accumulate(precisao[::-1])[::-1]
    degraus = np.diff(np.concatenate([[0.0], revocacao]))
    ap = float(np.sum(degraus * envelope))
    return ap, revocacao, envelope

# Detector sintetico sobre 10 "imagens": acerta ~85% dos alvos com
# jitter de localizacao e emite alguns falsos positivos por imagem.
rng = np.random.default_rng(SEMENTE)
gt_imgs, det_imgs = [], []
for _ in range(10):
    gts = []
    for _ in range(rng.integers(2, 5)):
        x, y = rng.uniform(5, 70, 2)
        w, h = rng.uniform(12, 25, 2)
        gts.append((x, y, x + w, y + h))
    dets = []
    for g in gts:
        if rng.uniform() < 0.85:            # alvo encontrado...
            j = rng.normal(0, 2.5, 4)       # ...com erro de localizacao
            dets.append((float(rng.uniform(0.55, 0.99)),
                         (g[0] + j[0], g[1] + j[1], g[2] + j[2], g[3] + j[3])))
    for _ in range(rng.integers(0, 3)):     # falsos positivos
        x, y = rng.uniform(5, 70, 2)
        w, h = rng.uniform(10, 22, 2)
        dets.append((float(rng.uniform(0.2, 0.6)), (x, y, x + w, y + h)))
    gt_imgs.append(gts); det_imgs.append(dets)

plt.figure(figsize=(7, 4.6))
for limiar, cor in [(0.5, "tab:blue"), (0.75, "tab:red")]:
    ap, rev, env = average_precision(gt_imgs, det_imgs, limiar_iou=limiar)
    plt.plot(rev, env, lw=2, color=cor,
             label=f"IoU >= {limiar}: AP = {ap:.3f}")
plt.xlabel("Revocacao"); plt.ylabel("Precisao (envelope)")
plt.title("Curva precisao-revocacao do detector: a AP e a area sob ela")
plt.legend(); plt.tight_layout(); plt.show()
print("mAP = media das AP sobre as classes (aqui, uma classe unica).")

**Observe:** o **mesmo detector** tem AP bem menor no limiar de IoU 0,75 — a métrica cobra a *qualidade da localização*, não apenas o acerto do rótulo. Por isso avaliações sérias reportam a mAP em **mais de um limiar**.

---

Do conceito à prática: a **Listagem 4.2** executa a inferência com um **YOLO pré-treinado** (pesos COCO). Em defesa, faz-se o *fine-tuning* sobre um conjunto rotulado do domínio operacional (ver Capítulo 3). A célula seguinte instala a biblioteca `ultralytics` — a única dependência extra do capítulo (no Colab, ~30 s; requer internet para baixar os pesos, ~6 MB):

In [ ]:
%pip install -q ultralytics

In [ ]:
# Listagem 4.2 - Inferencia com um detector YOLO pre-treinado.
from ultralytics import YOLO

# YOLO pre-treinado (pesos COCO). Em defesa, faz-se o fine-tuning
# sobre um conjunto rotulado do dominio operacional (ver Cap. 3).
modelo = YOLO("yolov8n.pt")

# Inferencia sobre uma imagem de exemplo, com limiar de confianca.
# (Substitua a URL por um quadro do seu proprio sensor.)
IMAGEM = "https://ultralytics.com/images/bus.jpg"
resultados = modelo(IMAGEM, conf=0.35, verbose=False)

# Extrai as deteccoes acima do limiar.
for r in resultados:
    for caixa in r.boxes:
        classe = int(caixa.cls[0])
        conf = float(caixa.conf[0])
        x1, y1, x2, y2 = caixa.xyxy[0].tolist()
        nome = modelo.names[classe]
        print(f"{nome:12s} conf={conf:.2f} "
              f"caixa=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})")

In [ ]:
# Visualizacao: as deteccoes sobre a imagem
anotada = resultados[0].plot()          # BGR, com caixas e rotulos
plt.figure(figsize=(7.5, 7.5))
plt.imshow(anotada[:, :, ::-1])         # BGR -> RGB
plt.axis("off")
plt.title("YOLO pre-treinado: caixas, classes e confiancas")
plt.tight_layout(); plt.show()

**Observe:** uma única passagem da rede devolveu todas as caixas, com classe e confiança — já **depois** da NMS interna do detector. O limiar `conf` é a mesma alavanca operacional do Capítulo 2: baixá-lo aumenta a revocação e o número de falsos positivos (Exercício 4.2).

---
## 4.4 Rastreamento multiobjetivo

Detectar objetos em quadros isolados não basta: a operação exige **seguir o mesmo objeto ao longo do tempo** — atribuir-lhe uma *identidade persistente*, estimar sua trajetória, contar quantos alvos distintos cruzaram a cena. Esse é o **rastreamento multiobjetivo** (*multi-object tracking*, MOT).

### 4.4.1 Rastreamento por detecção

O paradigma dominante é o **rastreamento por detecção**: em cada quadro, executa-se o detector; em seguida, **associam-se** as novas detecções às **faixas** (*tracks*) já existentes. A associação apoia-se em dois componentes:

- A **predição de movimento**, tipicamente por um **filtro de Kalman**, projeta onde cada faixa *deveria* estar no quadro atual, assumindo movimento suave.
- A **associação de dados** casa então detecções e predições, resolvendo um problema de **atribuição ótima** — minimizar o custo total do emparelhamento — pelo **algoritmo húngaro**, com o custo definido, por exemplo, como $1 - \mathrm{IoU}$ entre a detecção e a faixa predita.

Os algoritmos **SORT** (*Simple Online and Realtime Tracking*) e sua extensão **DeepSORT** — que acrescenta um descritor de aparência aprendido, para reidentificar objetos após oclusões — são as referências práticas. A **Listagem 4.3** implementa o coração do método:

> **✅ Boa prática** — O rastreamento é frágil onde a operação é mais difícil: sob **oclusão** (alvos que se cruzam ou se escondem) e em cenas densas, ocorrem **trocas de identidade** (*ID switches*) — a faixa de um alvo migra para outro. Duas defesas ajudam: um **descritor de aparência** (DeepSORT) para reidentificar após a oclusão, e a manutenção de **faixas "fantasma"** por alguns quadros, à espera do reaparecimento do alvo. Nenhuma delas elimina o problema; por isso, **contagens e trajetórias automáticas devem ser tratadas como estimativas sujeitas a erro**, não como verdades — sobretudo quando alimentam uma decisão.

In [ ]:
# Listagem 4.3 - Associacao de dados por algoritmo hungaro
# (rastreamento por deteccao).
from scipy.optimize import linear_sum_assignment

def associar(faixas, deteccoes, limiar_iou=0.3):
    # Casa deteccoes do quadro atual com as faixas preditas (Kalman),
    # minimizando o custo 1 - IoU pelo algoritmo hungaro.
    if len(faixas) == 0 or len(deteccoes) == 0:
        return [], list(range(len(deteccoes)))

    custo = np.zeros((len(faixas), len(deteccoes)))
    for i, f in enumerate(faixas):
        for j, d in enumerate(deteccoes):
            custo[i, j] = 1.0 - iou(f, d)   # reusa iou() da Listagem 4.1

    linhas, colunas = linear_sum_assignment(custo)

    pares, casadas = [], set()
    for i, j in zip(linhas, colunas):
        if 1.0 - custo[i, j] >= limiar_iou:   # emparelhamento valido
            pares.append((i, j))
            casadas.add(j)

    # deteccoes sem faixa correspondente iniciam novos alvos
    novas = [j for j in range(len(deteccoes)) if j not in casadas]
    return pares, novas

print("Funcao associar() definida: o coracao do rastreamento por deteccao.")

In [ ]:
# Um mini-SORT completo: Kalman + associacao hungara + gestao de faixas
class KalmanCV:
    # Filtro de Kalman com modelo de velocidade constante em 2D.
    def __init__(self, x, y):
        self.estado = np.array([x, y, 0.0, 0.0])       # [x, y, vx, vy]
        self.P = np.diag([10.0, 10.0, 100.0, 100.0])   # incerteza
        self.F = np.array([[1., 0., 1., 0.], [0., 1., 0., 1.],
                           [0., 0., 1., 0.], [0., 0., 0., 1.]])
        self.H = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.]])
        self.Q = np.eye(4) * 0.5                       # ruido do processo
        self.R = np.eye(2) * 4.0                       # ruido da medida

    def prever(self):
        self.estado = self.F @ self.estado
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.estado[:2]

    def corrigir(self, z):
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.estado = self.estado + K @ (np.asarray(z) - self.H @ self.estado)
        self.P = (np.eye(4) - K @ self.H) @ self.P

def caixa_de_centro(cx, cy, lado=12.0):
    m = lado / 2.0
    return (cx - m, cy - m, cx + m, cy + m)

def rastreia(deteccoes_por_quadro, limiar_iou=0.25, paciencia=3):
    # Mini-SORT: para cada quadro, prever -> associar -> atualizar.
    faixas, proximo_id, historico = {}, 0, []
    for q, dets in enumerate(deteccoes_por_quadro):
        ids = list(faixas.keys())
        preditas = [caixa_de_centro(*faixas[i]["kf"].prever()) for i in ids]
        caixas_det = [caixa_de_centro(cx, cy) for cx, cy in dets]

        pares, novas = associar(preditas, caixas_det, limiar_iou)

        casadas = set()
        for i_f, j_d in pares:
            faixas[ids[i_f]]["kf"].corrigir(dets[j_d])
            faixas[ids[i_f]]["perdidos"] = 0
            casadas.add(ids[i_f])
            historico.append((q, ids[i_f], *dets[j_d]))
        # faixas nao casadas: "fantasmas" por ate `paciencia` quadros
        for i in ids:
            if i not in casadas:
                faixas[i]["perdidos"] += 1
                if faixas[i]["perdidos"] > paciencia:
                    del faixas[i]
        # deteccoes sem faixa iniciam novos alvos
        for j in novas:
            faixas[proximo_id] = {"kf": KalmanCV(*dets[j]), "perdidos": 0}
            historico.append((q, proximo_id, *dets[j]))
            proximo_id += 1
    return historico, proximo_id

# Cena simulada: 4 alvos em velocidade constante; detector imperfeito
# (15% de falhas de deteccao e ruido de posicao) -- SEM identidades.
def simula_cena(n_quadros=40, p_falha=0.15, ruido=1.0, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    iniciais = np.array([[5., 10.], [5., 90.], [95., 20.], [50., 2.]])
    veloc = np.array([[2.0, 1.7], [2.0, -1.7], [-2.0, 1.4], [0.0, 2.1]])
    verdade, deteccoes = [], []
    for q in range(n_quadros):
        pos = iniciais + q * veloc
        verdade.append(pos.copy())
        dets = [tuple(p + rng.normal(0, ruido, 2)) for p in pos
                if rng.uniform() > p_falha]
        deteccoes.append(dets)
    return verdade, deteccoes

verdade, deteccoes = simula_cena()
historico, n_faixas = rastreia(deteccoes, paciencia=3)

plt.figure(figsize=(8, 6))
verdade_arr = np.array(verdade)
for a in range(4):
    plt.plot(verdade_arr[:, a, 0], verdade_arr[:, a, 1], "-",
             color="0.75", lw=5, zorder=0,
             label="trajetoria real" if a == 0 else None)
hist = np.array(historico)
for i in sorted(set(hist[:, 1].astype(int))):
    pts = hist[hist[:, 1] == i]
    plt.plot(pts[:, 2], pts[:, 3], "o-", ms=3, lw=1.2,
             label=f"faixa id={i}")
plt.gca().invert_yaxis()
plt.title(f"Rastreamento por deteccao: {n_faixas} faixas para 4 alvos reais")
plt.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

print(f"Alvos reais: 4 | faixas criadas: {n_faixas}")
print("Deteccoes brutas NAO tem identidade; as faixas a atribuem e mantem.")

**Observe:** o rastreador recebe apenas **pontos anônimos por quadro** — com falhas de detecção e ruído — e devolve **identidades persistentes**: cada alvo conserva o seu `id` do início ao fim, mesmo quando a detecção falha por alguns quadros (é a *paciência* das faixas fantasma segurando a identidade). Quando dois alvos se **cruzam**, a predição de movimento do Kalman é o que impede a troca: no cruzamento, a posição sozinha é ambígua, mas as *velocidades* não são.

O Exercício 4.5 explora o parâmetro `paciencia` — o equilíbrio entre **trocas de identidade** (paciência de menos) e **faixas espúrias** (paciência de mais).

---
## 4.5 Sensores de defesa: SAR, satélite e VANT

Os modelos até aqui foram descritos sobre imagens ópticas genéricas. A visão computacional para defesa, porém, opera sobre **sensores com características próprias**, que impõem cuidados específicos.

### 4.5.1 Radar de abertura sintética (SAR)

O **SAR** (*Synthetic Aperture Radar*) é um sensor **ativo** de micro-ondas: emite seu próprio sinal e mede o retorno. Sua grande virtude operacional é **independer da luz e do tempo** — enxerga à noite, através de nuvens e fumaça —, o que o torna indispensável em vigilância persistente. Em troca, a imagem SAR é radicalmente diferente da óptica: mede **reflectividade de micro-ondas**, não cor, e é corrompida por um ruído **multiplicativo** característico, o ***speckle***. Um detector treinado em imagens ópticas **não funciona sobre SAR sem adaptação**; exige pré-processamento próprio (filtragem de *speckle*) e, idealmente, treino sobre dados SAR. Aplicações típicas: detecção de embarcações em vigilância marítima e detecção de mudanças entre passagens sucessivas.

### 4.5.2 Imagens de satélite e de VANT

O imageamento de **satélite** oferece cobertura de área ampla e, frequentemente, múltiplas bandas espectrais além do visível. Seus desafios: resolução variável, cenas colossais (processamento em blocos) e **objetos muito pequenos** em relação ao quadro. O **VANT**, no extremo oposto, fornece imagem de alta resolução, baixa altitude e **tempo real** — o domínio tático por excelência, e o do *Project Maven*.

Atravessa todos esses sensores um problema comum, já anunciado no Capítulo 3: o **deslocamento de domínio** (*domain shift*) e a **escassez de rótulos**. Modelos treinados em um sensor, região ou estação degradam-se em outro; e rótulos operacionais confiáveis são caros e raros. As respostas são as do Módulo I — **transferência de aprendizado** a partir de modelos pré-treinados e **aumento de dados** — somadas ao rigor de **validar o modelo nas condições reais de emprego**, não nas de laboratório.

> **🛡️ Contexto de defesa** — A extensão territorial brasileira faz da vigilância por imagem uma **questão de soberania**. Sistemas de monitoramento de fronteiras, como o SISFRON do Exército, e o monitoramento da Amazônia por sensoriamento remoto ilustram a escala do problema: milhões de quilômetros quadrados a observar com recursos finitos. A visão computacional é, nesse contexto, um **multiplicador de capacidade** — prioriza a atenção humana sobre a fração relevante do imenso volume de dados. Vale aqui o princípio de **soberania tecnológica** do Capítulo 1: a dependência de modelos e de dados de sensoriamento estrangeiros é, ela própria, um risco operacional a ponderar.

A célula abaixo simula a diferença entre a mesma cena vista por um sensor óptico e por um SAR — e mostra o efeito da filtragem de *speckle*:

In [ ]:
# Optico vs. SAR: a mesma cena, fisicas diferentes
from scipy.ndimage import median_filter

rng = np.random.default_rng(SEMENTE)

# Cena "optica": mar + duas embarcacoes + faixa de costa
otica = rng.normal(0.30, 0.03, (120, 120))
otica[:, 95:] += 0.25                          # costa
for x, y, w, h in [(18, 40, 16, 6), (55, 75, 20, 7)]:
    otica[y:y + h, x:x + w] += 0.45
otica = np.clip(otica, 0, 1)

# Versao "SAR": reflectividade + ruido MULTIPLICATIVO (speckle)
reflect = otica * 0.8 + 0.1
speckle = rng.gamma(shape=1.0, scale=1.0, size=reflect.shape)
sar = np.clip(reflect * speckle, 0, 2.5)

# Pre-processamento tipico: filtragem de speckle (mediana)
sar_filtrado = median_filter(sar, size=5)

fig, eixos = plt.subplots(1, 3, figsize=(12.5, 4.4))
for eixo, img, titulo in [
        (eixos[0], otica, "sensor optico"),
        (eixos[1], sar, "SAR bruto: speckle multiplicativo"),
        (eixos[2], sar_filtrado, "SAR filtrado (mediana 5x5)")]:
    eixo.imshow(img, cmap="gray")
    eixo.set_title(titulo)
    eixo.axis("off")
plt.tight_layout(); plt.show()

print("Mesma cena, estatisticas de pixel radicalmente diferentes:")
print(f"  desvio-padrao optico:       {otica.std():.3f}")
print(f"  desvio-padrao SAR bruto:    {sar.std():.3f}")
print(f"  desvio-padrao SAR filtrado: {sar_filtrado.std():.3f}")
print("Um detector treinado no optico NAO transfere ao SAR sem adaptacao.")

**Observe:** o *speckle* não é um ruído aditivo suave — é **multiplicativo** e granular, e muda por completo a estatística dos pixels. Este é o *deslocamento de domínio* em sua forma mais visível; treinar e validar **no domínio de emprego** não é refinamento, é requisito.

---
## 4.6 Aplicação integrada: triagem para vigilância de área

Reunimos as peças em um problema de **vigilância de área ampla**. Um fluxo de imagens de sensor — de VANT ou de patrulha — precisa ser varrido em busca de **alvos de interesse** (embarcações, veículos, aeronaves), que devem ser **rastreados, contabilizados e sinalizados para revisão humana**. A tarefa, mais uma vez, não é substituir o analista, mas **priorizar sua atenção**; a métrica operacional decisiva é a **revocação da classe-alvo** — é grave deixar passar um alvo real.

A **Listagem 4.4** encadeia detecção e rastreamento em um fluxo contínuo, registrando os alvos de interesse por **identidade persistente**. O modo de rastreamento do YOLO integra, em uma chamada, o detector e um associador do tipo SORT — a mesma lógica da Listagem 4.3. No livro, a fonte é um vídeo de patrulha (`source="patrulha.mp4"`); aqui, simulamos a varredura do sensor com **recortes deslizantes** sobre a imagem de exemplo — o mesmo código se aplica a qualquer fluxo de quadros:

In [ ]:
# Listagem 4.4 (adaptada) - Pipeline de triagem: deteccao e rastreamento
# continuos para o analista.
import urllib.request
from PIL import Image

# Fluxo de quadros: "camera" varrendo a cena (recortes deslizantes).
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg",
                           "quadro_base.jpg")
base = np.array(Image.open("quadro_base.jpg"))
altura, largura = base.shape[:2]
lado = 640
y0 = (altura - lado) // 2
quadros = []
for passo in range(12):
    x0 = int(passo * (largura - lado) / 11)
    quadros.append(base[y0:y0 + lado, x0:x0 + lado].copy())

# Pipeline de vigilancia: detecta, rastreia e registra alvos de
# interesse, priorizando a atencao do analista sobre o fluxo.
modelo = YOLO("yolov8n.pt")

CLASSES_INTERESSE = {"bus", "truck", "car", "person"}   # no livro:
LIMIAR_CONF = 0.40                     # {"boat", "truck", "airplane", "car"}

registro, anotados = [], []
for quadro_id, quadro in enumerate(quadros):
    # modo track: deteccao + associacao entre quadros (rastreador SORT)
    r = modelo.track(quadro, conf=LIMIAR_CONF, persist=True,
                     verbose=False)[0]
    anotados.append(r.plot())          # guarda para visualizacao adiante
    if r.boxes is None or r.boxes.id is None:
        continue
    for caixa in r.boxes:
        nome = modelo.names[int(caixa.cls[0])]
        if nome not in CLASSES_INTERESSE:
            continue
        registro.append({
            "quadro": quadro_id,
            "id": int(caixa.id[0]),           # identidade persistente
            "classe": nome,
            "conf": round(float(caixa.conf[0]), 2),
        })

# Alvos unicos priorizados para revisao humana (controle humano -- Cap. 10).
ids_unicos = {reg["id"] for reg in registro}
print(f"Alvos de interesse rastreados: {len(ids_unicos)}")
print(f"Deteccoes registradas: {len(registro)}")

In [ ]:
# A identidade persiste enquanto a "camera" varre a cena
indices = [0, 4, 8, 11]
fig, eixos = plt.subplots(1, 4, figsize=(13.5, 3.8))
for eixo, i in zip(eixos, indices):
    eixo.imshow(anotados[i][:, :, ::-1])
    eixo.set_title(f"quadro {i}")
    eixo.axis("off")
plt.suptitle("Rastreamento no fluxo: cada alvo conserva o seu id entre quadros")
plt.tight_layout(); plt.show()

**Observe:** o mesmo objeto conserva o seu `id` de um quadro a outro — é a identidade persistente que permite **contar alvos únicos** (em vez de recontar o mesmo alvo a cada quadro) e construir trajetórias.

Como no capítulo anterior, o desfecho não é o número, mas a **decisão que ele apoia**. O sistema **tria**; o ser humano **decide**. Em detecção de alvos, essa fronteira é ainda mais sensível: um falso positivo confiante, ou um viés herdado dos dados de treino, pode induzir a erro de graves consequências. A avaliação por mAP, a **gestão explícita de falsos positivos** e a manutenção do **controle humano significativo** sobre a decisão não são refinamentos opcionais — são **condições de emprego responsável**, que o Capítulo 10 tratará como exigência normativa.

> **✏️ Experimente:** reduza `LIMIAR_CONF` para `0.15` e reexecute a Listagem 4.4. Observe o aumento do número de detecções — e pergunte-se quantas delas você, como analista, gostaria de revisar por hora de patrulha.

---
## 4.7 O caminho à frente

Este capítulo levou o aprendizado profundo do reconhecimento de imagens à **percepção operacional**: detectar, avaliar por mAP, rastrear no tempo e lidar com os sensores próprios da defesa. Com ele, cobrimos a metade **visual** da percepção. A outra metade é a **linguagem**.

O **Capítulo 5** trata do processamento de linguagem natural para inteligência — representação de texto, classificação temática, análise de sentimento e detecção de desinformação. O **Capítulo 6** chega aos modelos de linguagem de grande escala e à arquitetura *Transformer* que, não por acaso, hoje também redefine a própria visão computacional, aproximando os dois mundos em sistemas **multimodais**.

A disciplina persiste e se acumula: comparar contra uma *baseline*, medir pela métrica que reflete o custo operacional do erro, validar nas condições reais de emprego e manter o humano no centro da decisão. Cada capítulo a torna mais necessária — e, em percepção aplicada a alvos, **inegociável**.

## 📋 Resumo do capítulo

- A visão computacional distingue **classificação** (*o quê*), **detecção** (*o quê e onde*, por caixas delimitadoras) e **segmentação** (por *pixel*). Para a operação — contar, geolocalizar, rastrear — a **detecção** é o nível adequado.

- A **IoU** mede a qualidade da localização e é a base tanto da **avaliação** quanto da **supressão não-máxima**, que remove detecções duplicadas de um mesmo objeto.

- Os detectores de **duas etapas** (Faster R-CNN, com rede de proposição e caixas âncora) são mais precisos; os de **uma etapa** (YOLO) são mais rápidos e viabilizam a detecção em **tempo real**. A escolha é uma troca entre **acurácia e latência**.

- Detectores avaliam-se por **mAP** — a média, sobre as classes, da área sob a curva precisão–revocação —, **jamais por acurácia de classificação**.

- O **rastreamento multiobjetivo** segue alvos no tempo por *rastreamento por detecção*: predição de movimento (**Kalman**) e associação de dados (**algoritmo húngaro**), como em SORT e DeepSORT. Oclusão e cenas densas provocam **trocas de identidade**.

- Os sensores de defesa — **SAR** (ativo, todo-tempo, com *speckle*), **satélite** (área ampla) e **VANT** (tático, tempo real) — exigem tratamento próprio e enfrentam **deslocamento de domínio** e **escassez de rótulos**, mitigados por transferência de aprendizado e aumento de dados.

## ⚠️ Armadilhas comuns

1. **Avaliar um detector por acurácia de classificação.** A acurácia ignora a localização e o número variável de objetos. Reporte **mAP**, em um ou mais limiares de IoU, sempre.

2. **Aplicar a SAR um modelo treinado em imagens ópticas.** SAR mede micro-ondas, não cor, e carrega *speckle*. Sem pré-processamento e treino no domínio, o detector falha. O mesmo vale, em menor grau, para cada novo sensor, região ou estação — o *deslocamento de domínio*.

3. **Levar um detector a tempo real sem orçamento de latência.** Um modelo preciso, porém lento demais para o fluxo de quadros, é inútil em operação. Dimensione a arquitetura à latência disponível (e ao *hardware* embarcado do Módulo IV).

4. **Ajustar mal o limiar da NMS.** Um limiar alto demais funde alvos próximos em uma única caixa; baixo demais deixa duplicatas. Calibre-o conforme a densidade típica da cena.

5. **Confiar em contagens e trajetórias automáticas.** Oclusões e aglomerações geram trocas de identidade e faixas espúrias. Trate a saída do rastreador como **estimativa sujeita a erro**, não como verdade — sobretudo quando alimenta uma decisão.

6. **Ceder ao viés de automação na identificação de alvos.** Uma caixa confiante não é uma verdade, e o modelo herda os vieses de seus dados. Preserve o **controle humano significativo** sobre a decisão e gerencie explicitamente os falsos positivos (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 4.1** — Implemente e teste a função `iou` da Listagem 4.1 em três casos: caixas **idênticas**, caixas **disjuntas** e caixas com **metade de sobreposição**. Confirme que os resultados são, respectivamente, próximos de 1, iguais a 0 e coerentes com a fração sobreposta. Em duas linhas, explique por que a IoU é preferível à mera distância entre os centros das caixas.

In [ ]:
# Exercicio 4.1 - a funcao iou em tres casos de referencia
caixa = (10, 10, 50, 50)                       # 40 x 40

identica  = (10, 10, 50, 50)
disjunta  = (60, 60, 90, 90)
metade    = (30, 10, 70, 50)   # <--- ALTERE e verifique outros casos

for nome, outra in [("identica", identica), ("disjunta", disjunta),
                    ("metade sobreposta", metade)]:
    print(f"IoU caixa vs. {nome:18s} = {iou(caixa, outra):.3f}")

# Conferencia analitica do caso "metade": intersecao = 20x40 = 800;
# uniao = 1600 + 1600 - 800 = 2400; IoU esperada = 800/2400 = 0.333.

# Sua resposta (2 linhas):
# A IoU e preferivel a distancia entre centros porque ...

**Exercício 4.2** — Execute a célula abaixo (Listagem 4.2 com laço), variando o parâmetro `conf` em $\{0{,}1;\ 0{,}35;\ 0{,}7\}$ — e em $0{,}9$, para ver a revocação desabar. Relate como o **número de detecções** e a proporção de **falsos positivos** mudam com o limiar, e relacione o resultado à troca entre **precisão e revocação** do Capítulo 2. *(Requer a `ultralytics` instalada na Seção 4.3.)*

In [ ]:
# Exercicio 4.2 - o limiar de confianca como alavanca precisao/revocacao
from ultralytics import YOLO

modelo_ex = YOLO("yolov8n.pt")
IMAGEM = "https://ultralytics.com/images/bus.jpg"   # <--- ALTERE para
                                                    # uma imagem sua
for conf in [0.1, 0.35, 0.7, 0.9]:
    r = modelo_ex(IMAGEM, conf=conf, verbose=False)[0]
    n = len(r.boxes)
    escores = [round(float(c.conf[0]), 2) for c in r.boxes]
    print(f"conf = {conf:4.2f} -> {n:2d} deteccoes | escores: {escores}")

# Sua resposta (poucas linhas):
# 1) Efeito do limiar sobre o numero de deteccoes: ...
# 2) Relacao com precisao (limiar alto) vs. revocacao (limiar baixo): ...

**Exercício 4.3** — Aplique a função `nms` da Listagem 4.1 a um conjunto de caixas sobrepostas construído por você, variando o limiar entre 0,3 e 0,7. Descreva o efeito sobre o número de caixas mantidas e explique, com um exemplo, os **dois modos de falha**: fundir alvos distintos e reter duplicatas.

In [ ]:
# Exercicio 4.3 - os dois modos de falha da NMS
# Dois alvos PROXIMOS (caixas 0-2 no primeiro, 3-5 no segundo) --
# construa outros cenarios alterando as caixas abaixo.
caixas_ex = np.array([
    (20, 20, 50, 50), (24, 24, 54, 54),    # alvo A + duplicata (IoU 0.60)
    (27, 27, 57, 57), (29, 29, 59, 59),    # alvo B ENCOSTADO em A + duplicata
])                              # <--- ALTERE as caixas e observe
escores_ex = np.array([0.95, 0.80, 0.92, 0.78])

for limiar in [0.3, 0.4, 0.5, 0.6, 0.7]:
    mantidas = nms(caixas_ex, escores_ex, limiar=limiar)
    print(f"limiar {limiar:.1f} -> {len(mantidas)} caixas mantidas "
          f"(indices {[int(i) for i in mantidas]})")

# Sua resposta (poucas linhas):
# 1) Limiar BAIXO demais (ex.: 0.3) sobre alvos proximos: ...
# 2) Limiar ALTO demais (ex.: 0.7) sobre o mesmo alvo: ...

### Tático

**Exercício 4.4** *(dissertativo)* — Compare, em uma **tabela**, um detector de duas etapas (Faster R-CNN) e um de uma etapa (YOLO) segundo quatro critérios: **acurácia (mAP)**, **latência**, **adequação a *hardware* embarcado** e **facilidade de *fine-tuning***. Em seguida, recomende, justificando, um deles para (a) análise de imagens de satélite em retaguarda e (b) triagem em tempo real a bordo de um VANT.

*Responda em uma célula de texto abaixo.*

**Exercício 4.5** — Usando o mini-SORT da Seção 4.4, a célula abaixo varia a política de **faixas fantasma**: quando uma faixa não é casada em um quadro, ela é mantida por até `paciencia` quadros antes de descartada, à espera do reaparecimento do alvo. Execute com `paciencia` em $\{0, 3, 10\}$ e discuta, em poucas linhas, o efeito sobre as **trocas/fragmentações de identidade** e sobre o surgimento de **faixas espúrias**.

In [ ]:
# Exercicio 4.5 - a politica de faixas fantasma (parametro paciencia)
# Cena mais dificil: mais falhas de deteccao (30%) forcam o dilema.
verdade_ex, deteccoes_ex = simula_cena(n_quadros=40, p_falha=0.30,
                                       semente=1)

for paciencia in [0, 3, 10]:      # <--- ALTERE e compare
    _, n_faixas = rastreia(deteccoes_ex, paciencia=paciencia)
    print(f"paciencia = {paciencia:2d} -> {n_faixas:2d} faixas criadas "
          f"(alvos reais: 4)")

# Sua resposta (poucas linhas):
# 1) paciencia = 0: cada falha de deteccao quebra a faixa porque ...
# 2) paciencia alta demais: risco de faixas espurias/fantasmas porque ...
# 3) O valor adequado depende da taxa de falha do detector e de ...

**Exercício 4.6** *(dissertativo)* — Considere a detecção de embarcações em **imagens SAR** de vigilância marítima. Liste **três diferenças concretas** entre SAR e imagem óptica que impedem o uso direto de um detector óptico, e proponha, para cada uma, uma **medida de adaptação** (pré-processamento, aumento de dados específico ou estratégia de treino).

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 4.7** — Projete, em diagrama, um **pipeline completo de vigilância marítima de área ampla** que combine imagem de **satélite** (varredura ampla) e **VANT** (verificação de perto). Especifique os detectores em cada etapa, o ponto de passagem de uma para a outra, a métrica operacional e os **pontos de controle humano** na decisão de classificar um contato como alvo.

**Exercício 4.8** — Em um breve texto técnico (até duas páginas), discuta o problema do **deslocamento de domínio** na visão computacional para defesa: por que um detector validado em laboratório pode falhar em operação, e que estratégias de avaliação — e de **monitoramento contínuo em produção** — permitiriam detectar essa degradação a tempo. Relacione o argumento ao risco de *data shift* do Capítulo 3.

**Exercício 4.9** — Tome um conjunto público de imagens com anotações de detecção e conduza um **estudo completo**: faça o *fine-tuning* de um YOLO pré-treinado, reporte a **mAP em dois limiares de IoU**, compare-a à de uma *baseline* mais simples (por exemplo, o modelo pré-treinado sem ajuste) e redija um miniartigo de até três páginas — documentando os dados, as escolhas de treino, os resultados com variância e uma recomendação, incluindo os limites de generalização para imagens operacionais reais.

---

*Fim do Capítulo 4 — no Capítulo 5, a outra metade da percepção: o processamento de linguagem natural para inteligência.*